In [1]:
import sys; sys.path.insert(0, '..')
from fantasy.yahoo_client import get_my_roster, get_free_agents, get_current_matchup, get_league_categories, get_roster_by_team
from fantasy.nba_stats import get_player_stats, get_week_games_detail, get_matchup_factor, batch_get_player_stats
from fantasy.analysis import project_team_categories, project_player, classify_categories, usage_spike
from fantasy.llm import build_waiver_prompt, ask_gemini
import pandas as pd


In [2]:
matchup = get_current_matchup()
week_start, week_end = matchup['week_start'], matchup['week_end']
categories = get_league_categories()

def enrich(roster):
    with_stats = batch_get_player_stats(roster)
    enriched = []
    for p in with_stats:
        detail = get_week_games_detail(p['team_abbr'], week_start, week_end)
        mf = get_matchup_factor(p['team_abbr'], week_start, week_end)
        enriched.append({**p, 'games_remaining': detail['total'], 'games_detail': detail, 'matchup_factor': mf})
    return enriched

my_players = enrich(get_my_roster())
opp_players = enrich(get_roster_by_team(matchup['opponent_team_key']))

my_totals = project_team_categories(my_players)
opp_totals = project_team_categories(opp_players)

def to_league_cats(proj, cats):
    result = {}
    for cat in cats:
        if cat in proj:
            result[cat] = proj[cat]
        elif cat == 'FG%':
            result[cat] = proj['FGM'] / proj['FGA'] if proj.get('FGA', 0) > 0 else 0.0
        elif cat == 'FT%':
            result[cat] = proj['FTM'] / proj['FTA'] if proj.get('FTA', 0) > 0 else 0.0
        elif cat == '3PTM':
            result[cat] = proj.get('FG3M', 0.0)
        elif cat == 'ST':
            result[cat] = proj.get('STL', 0.0)
        elif cat == 'TO':
            result[cat] = proj.get('TOV', 0.0)
        elif cat == 'FGM/A':
            result[cat] = proj.get('FGM', 0.0)
        elif cat == 'FTM/A':
            result[cat] = proj.get('FTM', 0.0)
    return result

my_cats = to_league_cats(my_totals, categories)
opp_cats = to_league_cats(opp_totals, categories)
classification = classify_categories(my_cats, opp_cats)

weak_cats = [cat for cat, status in classification.items() if status in ('loss', 'tossup')]
print(f'Weakest categories: {weak_cats}')
print(f'\nMatchup breakdown:')
for cat, status in classification.items():
    print(f'  {cat:<8} {my_cats.get(cat,0):.2f} vs {opp_cats.get(cat,0):.2f}  → {status}')


In [3]:
# Typical weekly contribution per player (normalizes fit scores)
TYPICAL_WEEKLY = {
    'PTS': 55, 'REB': 18, 'AST': 12, '3PTM': 5,
    'ST': 3, 'BLK': 2, 'TO': 6, 'FGM/A': 14, 'FTM/A': 8, 'FG%': 0.47, 'FT%': 0.78,
}

fas = get_free_agents(count=200)

scored = []
for fa in fas:
    stats = get_player_stats(fa['name'])
    detail = get_week_games_detail(fa['team_abbr'], week_start, week_end)
    mf = get_matchup_factor(fa['team_abbr'], week_start, week_end)
    games = detail['total']
    if stats is None:
        continue
    proj = project_player(stats, detail, fa.get('status', ''), matchup_factor=mf)
    fa_cats = to_league_cats(proj, categories)
    usp = usage_spike(stats)
    # Fit score: normalized contribution in weak categories + small bonus for rising usage
    fit = 0.0
    for c in weak_cats:
        typical = TYPICAL_WEEKLY.get(c, 1.0)
        val = fa_cats.get(c, 0)
        if c in ('TO', 'TOV'):
            fit += (typical - val) / typical
        else:
            fit += val / typical
    fit += max(0, usp) * 0.05  # small bonus for rising usage
    scored.append({**fa, 'games_remaining': games, 'games_detail': detail,
                   'projected': proj, 'fit_score': fit, 'usg_spike': usp,
                   'matchup_factor': round(mf, 3)})

scored.sort(key=lambda x: x['fit_score'], reverse=True)
print(pd.DataFrame([{
    'Player': p['name'], 'Pos': p['position'], 'Status': p['status'] or 'Active',
    'G': p['games_remaining'], 'Fit': round(p['fit_score'], 2),
    'Matchup': p['matchup_factor'], 'USG%Δ': f"+{p['usg_spike']:.1f}" if p['usg_spike'] > 0 else f"{p['usg_spike']:.1f}"
} for p in scored[:30]]).to_string(index=False))


In [4]:
import os
from dotenv import load_dotenv
load_dotenv('/mnt/c/Users/louis/Documents/dev/.env', override=True)

prompt = build_waiver_prompt(scored, weak_cats)
advice = ask_gemini(prompt)
print('\n=== WAIVER WIRE PICKUPS ===\n')
print(advice)
